# 03b — Comparison of machine learning algorithms for the hybrid model

Same features as model D of notebook 03 (MZA-FAO features + altitude, DTA, latitude and longitude), same
target (log yield anomaly relative to the technology trend and municipality effect, estimated in each
fold) and the same leave-one-year-out validation (23 folds). Each algorithm is tested in up to three
configurations:

- **default**: library defaults (as commonly used in yield studies);
- **shallow**: shallow and regularized, with monotonic constraints where the library supports them;
- **nested**: hyperparameters chosen by a grid evaluated on 5 blocks of years *within* the training set of
  each outer fold, so the choice never sees the test year.

The ensembles of the best single algorithm with the monotonic gradient boosting are evaluated at the
end. Runtime: about 2.5 h on 4 cores (nested SVR dominates; SVR predicts only ~1,400 rows/s).

In [1]:
import itertools
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from joblib import Parallel, delayed
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler
from sklearn.svm import SVR

RESULTADOS = Path.cwd().parent / "resultados"
SEMENTE = 42
base = pd.read_parquet(RESULTADOS / "base_treino_hibrido.parquet")  # written by notebook 03
VAR_ESTATICO = ["altitude_m", "dta_mm_m", "latitude", "longitude"]
VAR_MZA = {"EC": 1, "ISNA_vegetativo": 1, "ISNA_floracao": 1, "ISNA_enchimento": 1, "DEF_floracao": -1,
           "frio_reprod": -1, "geada_reprod": -1, "calor_floracao": -1}
X_COLS = list(VAR_MZA) + VAR_ESTATICO
MONO = list(VAR_MZA.values()) + [0] * len(VAR_ESTATICO)
print(base.shape, base["ano"].min(), base["ano"].max())

(6806, 94) 2003 2025


In [2]:
def tendencia_e_municipio(treino):
    reg = LinearRegression().fit(treino[["ano"]], treino["log_y"])
    efeito = (treino["log_y"] - reg.predict(treino[["ano"]])).groupby(treino["cod_ibge"]).mean()
    return reg, efeito


def base_prevista(df, reg, efeito):
    return reg.predict(df[["ano"]]) + df["cod_ibge"].map(efeito).fillna(0).to_numpy()


def hgb(**k):
    return HistGradientBoostingRegressor(random_state=SEMENTE, **k)


RASO = dict(max_depth=3, learning_rate=0.05, max_iter=300, min_samples_leaf=100, l2_regularization=1.0)
MOD = {
    "HGB_shallow_monotonic (E)": lambda: hgb(monotonic_cst=MONO, **RASO),
    "HGB_shallow (D)": lambda: hgb(**RASO),
    "HGB_default": lambda: hgb(),
    "XGB_default": lambda: xgb.XGBRegressor(n_jobs=1, random_state=SEMENTE),
    "XGB_shallow_monotonic": lambda: xgb.XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, min_child_weight=100,
                                                      subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                                                      monotone_constraints=tuple(MONO), n_jobs=1, random_state=SEMENTE),
    "LGBM_default": lambda: lgb.LGBMRegressor(n_jobs=1, random_state=SEMENTE, verbose=-1),
    "LGBM_shallow_monotonic": lambda: lgb.LGBMRegressor(n_estimators=300, max_depth=3, num_leaves=8, learning_rate=0.05,
                                                        min_child_samples=100, subsample=0.8, subsample_freq=1,
                                                        colsample_bytree=0.8, reg_lambda=1.0, monotone_constraints=MONO,
                                                        n_jobs=1, random_state=SEMENTE, verbose=-1),
    "RF_default": lambda: RandomForestRegressor(300, n_jobs=1, random_state=SEMENTE),
    "RF_shallow": lambda: RandomForestRegressor(300, max_depth=6, min_samples_leaf=50, max_features=0.5, n_jobs=1, random_state=SEMENTE),
    "ET_default": lambda: ExtraTreesRegressor(300, n_jobs=1, random_state=SEMENTE),
    "ET_shallow": lambda: ExtraTreesRegressor(300, max_depth=8, min_samples_leaf=50, max_features=0.7, n_jobs=1, random_state=SEMENTE),
    "Ridge_splines": lambda: make_pipeline(StandardScaler(), SplineTransformer(n_knots=5, degree=3), RidgeCV(alphas=np.logspace(-2, 4, 20))),
    "MLP": lambda: make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32, 16), alpha=1.0, early_stopping=True,
                                                                max_iter=500, random_state=SEMENTE)),
    "SVR_fixed": lambda: make_pipeline(StandardScaler(), SVR(C=0.3, epsilon=0.05, gamma="scale")),
}
GRADES = {
    "HGB_monotonic_nested": (lambda p: hgb(monotonic_cst=MONO, l2_regularization=1.0, **p),
                             [dict(max_depth=d, learning_rate=lr, max_iter=it, min_samples_leaf=ml)
                              for d, lr, it, ml in itertools.product([2, 3, 5], [0.03, 0.1], [150, 400], [50, 200])]),
    "LGBM_monotonic_nested": (lambda p: lgb.LGBMRegressor(subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                                                          monotone_constraints=MONO, n_jobs=1, random_state=SEMENTE, verbose=-1, **p),
                              [dict(n_estimators=it, max_depth=d, num_leaves=2 ** d, learning_rate=lr, min_child_samples=ml)
                               for d, lr, it, ml in itertools.product([2, 3, 5], [0.03, 0.1], [150, 400], [50, 200])]),
    "SVR_nested": (lambda p: make_pipeline(StandardScaler(), SVR(epsilon=0.05, **p)),
                   [dict(C=c, gamma=g) for c in (0.1, 0.3, 1.0, 3.0) for g in ("scale", 0.05, 0.2)]),
}

In [3]:
def dobra(ano):
    tr, te = base["ano"] != ano, base["ano"] == ano
    reg, ef = tendencia_e_municipio(base[tr])
    b_te = base_prevista(base[te], reg, ef)
    an = (base.loc[tr, "log_y"] - base_prevista(base[tr], reg, ef)).to_numpy()
    Xtr, Xte = base.loc[tr, X_COLS].to_numpy(), base.loc[te, X_COLS].to_numpy()
    out, escolha, tempo = {}, {}, {}
    for nome, f in MOD.items():
        t0 = time.time()
        out[nome] = b_te + f().fit(Xtr, an).predict(Xte)
        tempo[nome] = time.time() - t0
    anos_tr = base.loc[tr, "ano"].to_numpy()
    blocos = np.array_split(np.unique(anos_tr), 5)  # inner folds: blocks of training years
    for nome, (f, grade) in GRADES.items():
        t0 = time.time()
        erro = []
        for p in grade:
            e = 0.0
            for bl in blocos:
                m = np.isin(anos_tr, bl)
                e += ((f(p).fit(Xtr[~m], an[~m]).predict(Xtr[m]) - an[m]) ** 2).sum()
            erro.append(e)
        melhor = grade[int(np.argmin(erro))]
        escolha[nome] = melhor
        out[nome] = b_te + f(melhor).fit(Xtr, an).predict(Xte)
        tempo[nome] = time.time() - t0
    return te.to_numpy(), out, escolha, tempo


res = Parallel(n_jobs=4)(delayed(dobra)(a) for a in sorted(base["ano"].unique()))
oof = base[["cod_ibge", "ano", "rendimento_kg_ha", "log_y"]].copy()
escolhas, tempos = {}, {}
for te, out, esc, t in res:
    for n_, v in out.items():
        oof.loc[te, n_] = v
    for n_, v in esc.items():
        escolhas.setdefault(n_, []).append(str(v))
    for n_, v in t.items():
        tempos[n_] = tempos.get(n_, 0) + v
for n_, v in escolhas.items():
    print(n_, pd.Series(v).value_counts().head(2).to_dict())

HGB_monotonic_nested {"{'max_depth': 5, 'learning_rate': 0.03, 'max_iter': 150, 'min_samples_leaf': 200}": 21, "{'max_depth': 3, 'learning_rate': 0.03, 'max_iter': 150, 'min_samples_leaf': 50}": 1}
LGBM_monotonic_nested {"{'n_estimators': 150, 'max_depth': 5, 'num_leaves': 32, 'learning_rate': 0.03, 'min_child_samples': 200}": 18, "{'n_estimators': 150, 'max_depth': 5, 'num_leaves': 32, 'learning_rate': 0.03, 'min_child_samples': 50}": 5}
SVR_nested {"{'C': 0.3, 'gamma': 0.05}": 22, "{'C': 0.1, 'gamma': 0.05}": 1}


In [4]:
oof["Ensemble SVR_nested + E"] = oof[["SVR_nested", "HGB_shallow_monotonic (E)"]].mean(axis=1)
oof["Ensemble HGB_nested + SVR_nested"] = oof[["SVR_nested", "HGB_monotonic_nested"]].mean(axis=1)
mods = [c for c in oof.columns if c not in ("cod_ibge", "ano", "rendimento_kg_ha", "log_y")]


def metricas(c):
    y, p = oof["rendimento_kg_ha"], np.exp(oof[c])
    anual = oof.assign(p=p).groupby("ano")[["rendimento_kg_ha", "p"]].mean()
    return {"R2": r2_score(y, p), "RMSE_kg_ha": float(np.sqrt(np.mean((y - p) ** 2))),
            "R2_annual": r2_score(anual["rendimento_kg_ha"], anual["p"])}


tab = pd.DataFrame({c: metricas(c) for c in mods}).T
tab["time_s"] = pd.Series(tempos)
anos_u = np.sort(oof["ano"].unique())
ix = np.searchsorted(anos_u, np.random.default_rng(SEMENTE).choice(anos_u, (2000, len(anos_u))))
cont = oof.groupby("ano").size().to_numpy()


def rmse_boot(c):
    e = ((oof["rendimento_kg_ha"] - np.exp(oof[c])) ** 2).groupby(oof["ano"]).sum().to_numpy()
    return np.sqrt(e[ix].sum(1) / cont[ix].sum(1))


ref = rmse_boot("HGB_shallow_monotonic (E)")
for c in mods:
    d = rmse_boot(c) - ref
    tab.loc[c, ["dRMSE_vs_E", "CI95_low", "CI95_high"]] = [np.median(d), *np.percentile(d, [2.5, 97.5])]
    tab.loc[c, "P_better_than_E"] = np.mean(d < 0)
tab.sort_values("RMSE_kg_ha").to_csv(RESULTADOS / "comparacao_algoritmos.csv")
tab.sort_values("RMSE_kg_ha").round(3)

,R2,RMSE_kg_ha,R2_annual,time_s,dRMSE_vs_E,CI95_low,CI95_high,P_better_than_E
Ensemble HGB_nested + SVR_nested,0.437,1052.078,0.681,NaN,-39.411,-75.337,-7.253,0.992
SVR_nested,0.436,1052.950,0.718,4137.749,-37.671,-91.304,6.221,0.950
Ensemble SVR_nested + E,0.435,1054.039,0.688,NaN,-37.366,-62.752,-15.566,0.999
SVR_fixed,0.415,1072.395,0.708,75.098,-19.525,-77.700,32.180,0.750
HGB_monotonic_nested,0.405,1081.823,0.626,1140.730,-10.678,-34.930,17.654,0.774
LGBM_monotonic_nested,0.403,1083.366,0.616,645.960,-9.278,-32.224,17.978,0.752
HGB_shallow (D),0.403,1083.899,0.656,9.252,-6.875,-44.399,27.178,0.660
LGBM_shallow_monotonic,0.398,1088.042,0.629,5.143,-3.773,-9.475,1.393,0.922
XGB_shallow_monotonic,0.395,1090.951,0.629,6.213,-0.954,-6.076,4.123,0.646
HGB_shallow_monotonic (E),0.394,1091.816,0.634,9.546,0.000,0.000,0.000,0.000
